# CENTRALIZED MODEL: FASTQ Features + Hand-Engineered Features

This model leverages both raw FASTQ-derived features and hand-engineered metrics for robust classification.

### Workflow
1. **Feature Extraction**

   - Convert compressed mgb files to .fastq files
   - Extract features from FASTQ files.  
   - Engineer additional custom features to capture sequence patterns, quality, and composition.

3. **Model Training**
   - Tune Lightgbm using optuna
   - Train 10 LightGBM models using 10-fold cross-validation.  
   - Local CV performance: **0.032**.  
   - Track carbon emissions during training; lowest footprint achieved.

5. **Feature Importance**  
   - Identify and visualize the most influential features.

6. **Ensembling & Inference**  
   - Average predictions from the 10 models to obtain final probabilities on the test set.  
   - No post-processing applied.

### Performance
- **Public LB:** 0.001306302  
- **Test set:** 0.023792115


# Install & Imports

In [1]:
from IPython.display import clear_output

In [2]:
!pip install uv \
    Bio \
    codecarbon \
    #shap==0.48.0 # --force-reinstall
clear_output()

In [5]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 15.3 MB/s eta 0:00:00


In [6]:
import os
import math
import shap
import optuna
import random
import warnings
import kagglehub
import numpy as np
import pandas as pd
from Bio import SeqIO
import lightgbm as lgb
import importlib.metadata
from itertools import product
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import log_loss
from scipy.stats import entropy#, gini
from codecarbon import EmissionsTracker
from sklearn.preprocessing import LabelEncoder
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import StratifiedKFold

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Environment

In [7]:
# List of packages you actually use
packages = [
    "shap",
    "optuna",
    "numpy",
    "pandas",
    "biopython",  # for Bio.SeqIO
    "lightgbm",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "codecarbon"
]

with open("requirements.txt", "w") as f:
    for pkg in packages:
        try:
            version = importlib.metadata.version(pkg)
            f.write(f"{pkg}=={version}\n")
        except importlib.metadata.PackageNotFoundError:
            # package not installed
            f.write(f"{pkg}\n")

In [8]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 100)

In [12]:
from codecarbon import EmissionsTracker  # (or EmissionTracker in older versions)

tracker = EmissionsTracker(
)
tracker.start()

[codecarbon WARNING @ 05:18:23] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 05:18:23] [setup] RAM Tracking...
[codecarbon INFO @ 05:18:23] [setup] CPU Tracking...
[codecarbon WARNING @ 05:18:24] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 05:18:24] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 05:18:24] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon WARNING @ 05:18:24] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 05:18:24] [setup] GPU Tracking...
[codecarbon INFO @ 05:18:24] No GPU found.
[codecarbon INFO @ 05:18:24] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
               

In [9]:
le = LabelEncoder()

- If you want to extract the raw features change Extract param to True. The train will take 4hrs to extract while test will take 2 hours. For now I have **train_more_feats.csv** and **test_more_feats.csv** ready for you,if you would not like to extract the features yourself.
-  Also, if you wanna tune the model like I did (takes roughly 15 minutes)

In [10]:
Params={'Tune':False,
       'Extract': False,
       }

# Read Data

In [ ]:
path = kagglehub.dataset_download("noob786/secondbatchoffastqfiles")
path = kagglehub.dataset_download("juliusmwangi/microbiome-classification-data")
path = kagglehub.dataset_download("noob786/mpeg-g-microbiomeclassificationconvertedfastqfiles")

In [ ]:
for folder, _, files in os.walk('microbiome-classification-data/'):
    for f in files:
        print(os.path.join(folder, f))

/kaggle/input/microbiome-classification-data/cytokine_profiles.csv
/kaggle/input/microbiome-classification-data/Viome_microbiome_metadata.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/Train_Subjects.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/Train copy.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/SampleSubmission.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/train.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/test.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/Test copy.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/test-copy.csv
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/Starter_NB_Part1_Decompressing_MPEG_G_files.ipynb
/kaggle/input/microbiome-classification-data/Microbiome Classification Data/Starter_NB_Part2_Model

In [21]:
csv_path ="/content"    # PATH TO VARIOUS CSVS

test_base_dirs = "/content/drive/Shareddrives/ZINDI Data Science/1. Competitions/MPEG-G (Phillips)/Solutions: #1 Microbiome Classification/uncompressed_data/TestFiles/TestFiles"

train_base_dirs = [
    "/content/drive/Shareddrives/ZINDI Data Science/1. Competitions/MPEG-G (Phillips)/Solutions: #1 Microbiome Classification/uncompressed_data/archive (1)/TrainFiles"
]

In [22]:
# Read in csv files
train_df = pd.read_csv("/content/Train.csv")
test_df = pd.read_csv(csv_path + "/Test.csv")
train_subjects_df = pd.read_csv(csv_path + "/Train_Subjects.csv")
ss = pd.read_csv(csv_path + "/SampleSubmission.csv")
display(
       test_df.shape,
       test_df.head(2),
        ss.shape,
       ss.head(2),
       train_df.shape,
       train_df.head(2),
       train_subjects_df.shape,
       train_subjects_df.head(2),
)

(1068, 1)

,filename
0,ID_YBNOYC.mgb
1,ID_HPVLUO.mgb


(1068, 5)

,filename,Mouth,Nasal,Skin,Stool
0,ID_YBNOYC.mgb,0,0,0,0
1,ID_HPVLUO.mgb,0,0,0,0


(2901, 4)

,filename,SampleType,SubjectID,SampleID
0,ID_LETPJN.mgb,Stool,Subject_BCUNIB,Sample_AFTIWE
1,ID_NTDGIW.mgb,Stool,Subject_UDAXIH,Sample_JQJVNK


(66, 17)

,SubjectID,FPG_Mean,FPG_class,IRIS,SSPG,FPG,SSPG.Date,Class,Gender,Ethnicity,Adj.age,BMI,OGTT,OGTT_Class,Longitudinal.HbA1C.Group,A1C_Class,Family
0,Subject_UDAXIH,1.274432,Diabetes,IS,91.5,131.75,8/7/14,Diabetic,M,C,59.48,21.47,2.245,Diabetes,6. Variable Diabetic-PreDM (n = 8),6.VDP,NaN
1,Subject_NHOSIZ,0.915833,Normal,Unknown,NaN,NaN,NaN,Prediabetic,M,C,61.17,27.06,1.005,Normal,3. PreDM-to-Normal (n = 10),3.PN,NaN


In [23]:
test_df["filename"] = test_df["filename"].str.replace(".mgb", ".fastq", regex=False)
train_df["filename"] = train_df["filename"].str.replace(".mgb", ".fastq", regex=False)

# FASTQ Feature Extraction for Modeling

We extract a comprehensive set of features from FASTQ files to capture **read-level, compositional, complexity, and quality characteristics** for downstream modeling. Features include:

### 1. Read-Level Metrics
- **Number of reads (`num_reads`)** – indicates sequencing depth/coverage.  
- **Average read length (`avg_read_len`)** – reflects sequencing quality and library preparation.  
- **Read length variability (`read_len_std`, `read_len_min`, `read_len_max`)** – captures heterogeneity, edge cases, or mixed populations.  

### 2. GC Content & Nucleotide Composition
- **GC content (`gc_content`)** – species- or condition-specific genomic signature.  
- **GC variability (`gc_std`)** – detects sequencing bias or heterogeneity.  
- **Nucleotide proportions (`A`, `T`, `G`, `C`)** – capture composition biases.  

### 3. Sequence Complexity
- **Average Shannon entropy (`avg_entropy`)** – measures sequence diversity vs. repetitiveness; flags low-complexity or contaminated samples.  

### 4. Quality Metrics
- **Average base quality (`avg_quality`)** – overall sequencing reliability.  
- **Quality score variability (`std_quality`)** – flags inconsistent runs or sample prep issues.  

### 5. K-mer Frequencies
- **Dinucleotide and trinucleotide frequencies (`<kmer>_freq`)** – normalized counts for all possible k-mers, capturing sequence motifs and patterns.  

✅ These features provide a **robust statistical and biological foundation** for modeling, encoding sequencing depth, variability, composition, complexity, quality, and motifs.


In [24]:
def shannon_entropy(seq):
    counts = Counter(seq)
    probs = [c/len(seq) for c in counts.values() if len(seq) > 0]
    return -sum(p*math.log2(p) for p in probs)

def extract_features_from_fastq(path, k_values=[2,3]):
    total_reads = 0
    total_bases = 0
    sum_read_len = 0
    sum_read_len_sq = 0
    gc_total = 0
    gc_frac_sum = 0
    sum_entropy = 0
    quality_sum = 0
    quality_sq_sum = 0
    quality_count = 0

    nt_counter = Counter()
    kmer_counters = {k: Counter() for k in k_values}

    for record in SeqIO.parse(path, "fastq"):
        seq = str(record.seq)
        n = len(seq)
        if n == 0:
            continue

        total_reads += 1
        total_bases += n
        sum_read_len += n
        sum_read_len_sq += n**2

        gc = seq.count("G") + seq.count("C")
        gc_total += gc
        gc_frac_sum += gc / n

        nt_counter.update(seq)
        ent = shannon_entropy(seq)
        sum_entropy += ent

        q = record.letter_annotations["phred_quality"]
        quality_sum += sum(q)
        quality_sq_sum += sum(v*v for v in q)
        quality_count += len(q)

        for k in k_values:
            kmer_counters[k].update(seq[i:i+k] for i in range(n-k+1))

    if total_reads == 0:
        return None

    feats = {
        "num_reads": total_reads,
        "avg_read_len": sum_read_len / total_reads,
        "read_len_std": math.sqrt(sum_read_len_sq/total_reads - (sum_read_len/total_reads)**2),
        "read_len_min": sum_read_len/total_reads if total_reads==1 else None,  # optional
        "read_len_max": None,  # skipped for streaming (could track separately)
        "gc_content": gc_total / total_bases,
        "gc_std": np.std([gc/l for gc,l in zip([gc_total],[total_bases])]),  # can approximate better
        "A": nt_counter["A"] / total_bases,
        "T": nt_counter["T"] / total_bases,
        "G": nt_counter["G"] / total_bases,
        "C": nt_counter["C"] / total_bases,
        "avg_entropy": sum_entropy / total_reads,
        "avg_quality": quality_sum / quality_count if quality_count else 0,
        "std_quality": math.sqrt(quality_sq_sum/quality_count - (quality_sum/quality_count)**2) if quality_count else 0,
    }

    for k in k_values:
        total_kmers = sum(kmer_counters[k].values())
        for mer in product("ACGT", repeat=k):
            mer = "".join(mer)
            feats[f"{mer}_freq"] = (
                kmer_counters[k][mer] / total_kmers if total_kmers else 0
            )

    return feats

In [25]:
### TEST
def process_test_row(row):
    fpath = os.path.join(test_base_dirs, row["filename"])
    feats = extract_features_from_fastq(fpath)
    if feats:
        feats["filename"] = row["filename"]
        return feats
    return None

In [26]:
%%time

# 2hrs 6gb cpu desktop

if Params['Extract']:
    #test_df = test_df.iloc[:5]  # test with fewer rows first

    with ProcessPoolExecutor() as ex:
        results = list(ex.map(process_test_row, [row for _, row in test_df.iterrows()]))

    test_features = [r for r in results if r]
    df_test = pd.DataFrame(test_features)

    df_test["ID"] = df_test["filename"].str.split(".").str[0]
    df_test.sort_values(by="ID", inplace=True)
    df_test.to_csv("test_more_feats.csv", index=False)
    display(df_test)

else:
    df_test = pd.read_csv('/kaggle/input/microbiome-classification-data/MoreFeatures/test_more_feats.csv')
    display(df_test)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/microbiome-classification-data/MoreFeatures/test_more_feats.csv'

In [ ]:
### TRAIN
def process_train_row(row):
    fname = row["filename"]

    # Normalize train_base_dirs into a list, even if it's a string
    bases = train_base_dirs if isinstance(train_base_dirs, (list, tuple)) else [train_base_dirs]

    fpath = None
    for base in bases:
        candidate = os.path.join(base, fname)
        if os.path.exists(candidate):
            fpath = candidate
            break

    if fpath is None:
        return None  # file not found in any base dir

    feats = extract_features_from_fastq(fpath)
    if feats:
        feats["filename"] = fname
        return feats
    return None

In [ ]:
%%time
# 3HRS  57MINS

if Params['Extract']:
    #train_df=train_df[train_df['filename'].isin(['ID_QEUWDL.fastq','ID_AAFNOT.fastq'])] #for testing

    train_features = []
    with ProcessPoolExecutor() as ex:
        results = list(ex.map(process_train_row, [row for _, row in train_df.iterrows()]))

    train_features = [r for r in results if r]  # filter out None
    df_train = pd.DataFrame(train_features)

    df_train['ID']=df_train['filename'].apply(lambda x: x.split('.')[:1][0])
    df_train.sort_values(by='ID',inplace=True)
    df_train.to_csv("train_more_feats.csv",index=False)
    display(df_train)
else:
    df_train = pd.read_csv('/kaggle/input/microbiome-classification-data/MoreFeatures/train_more_feats.csv')
    display(df_train)

,num_reads,avg_read_len,read_len_std,read_len_min,read_len_max,gc_content,gc_std,A,T,G,C,avg_entropy,avg_quality,std_quality,AA_freq,AC_freq,AG_freq,AT_freq,CA_freq,CC_freq,CG_freq,CT_freq,GA_freq,GC_freq,GG_freq,GT_freq,TA_freq,TC_freq,TG_freq,TT_freq,AAA_freq,AAC_freq,AAG_freq,AAT_freq,ACA_freq,ACC_freq,ACG_freq,ACT_freq,AGA_freq,AGC_freq,AGG_freq,AGT_freq,ATA_freq,ATC_freq,ATG_freq,ATT_freq,CAA_freq,CAC_freq,CAG_freq,CAT_freq,CCA_freq,CCC_freq,CCG_freq,CCT_freq,CGA_freq,CGC_freq,CGG_freq,CGT_freq,CTA_freq,CTC_freq,CTG_freq,CTT_freq,GAA_freq,GAC_freq,GAG_freq,GAT_freq,GCA_freq,GCC_freq,GCG_freq,GCT_freq,GGA_freq,GGC_freq,GGG_freq,GGT_freq,GTA_freq,GTC_freq,GTG_freq,GTT_freq,TAA_freq,TAC_freq,TAG_freq,TAT_freq,TCA_freq,TCC_freq,TCG_freq,TCT_freq,TGA_freq,TGC_freq,TGG_freq,TGT_freq,TTA_freq,TTC_freq,TTG_freq,TTT_freq,filename,label,ID
0,42840,124.5,0.5,NaN,NaN,0.552629,0.0,0.182194,0.265177,0.268545,0.284083,1.920269,33.814612,5.260114,0.042977,0.042083,0.053192,0.042267,0.048329,0.086290,0.077074,0.074589,0.053084,0.072132,0.076118,0.068617,0.039265,0.081836,0.064273,0.077874,0.011399,0.009149,0.013774,0.008986,0.003884,0.015486,0.012330,0.010724,0.008209,0.013395,0.024267,0.007750,0.007140,0.013177,0.002331,0.019952,0.002524,0.018687,0.016994,0.010508,0.026537,0.022126,0.012268,0.026061,0.006587,0.031601,0.012660,0.026838,0.002638,0.025675,0.020815,0.022066,0.017881,0.004217,0.014414,0.014143,0.006709,0.012391,0.028534,0.025073,0.020147,0.016368,0.021112,0.019107,0.015467,0.013305,0.020158,0.020177,0.011515,0.010369,0.008445,0.008975,0.011589,0.032926,0.024570,0.013337,0.018514,0.011355,0.018700,0.015479,0.010335,0.030344,0.021493,0.016315,ID_AAFNOT.fastq,Mouth,ID_AAFNOT
1,42814,400.0,0.0,NaN,NaN,0.539853,0.0,0.268928,0.191218,0.316762,0.223092,1.970735,36.432308,4.110366,0.068139,0.069039,0.075684,0.055772,0.059831,0.050969,0.066888,0.045795,0.090766,0.072336,0.104082,0.049873,0.050760,0.031305,0.068506,0.040255,0.012121,0.025302,0.019065,0.011652,0.018307,0.014046,0.025362,0.011469,0.018427,0.018038,0.021356,0.017933,0.014461,0.006695,0.020634,0.013861,0.015172,0.014303,0.016626,0.013879,0.013895,0.010529,0.012493,0.014112,0.016406,0.015687,0.025184,0.009728,0.011870,0.006757,0.017628,0.009548,0.023510,0.020072,0.025770,0.021247,0.022142,0.017390,0.019544,0.013428,0.027356,0.025066,0.032572,0.019173,0.013679,0.010128,0.016871,0.008967,0.017470,0.009520,0.014413,0.009081,0.005637,0.009132,0.009655,0.006901,0.026414,0.013726,0.025222,0.003164,0.010878,0.007803,0.013543,0.007980,ID_AAXPTO.fastq,Nasal,ID_AAXPTO
2,21794,400.0,0.0,NaN,NaN,0.543314,0.0,0.264808,0.191878,0.318555,0.224759,1.971770,36.337955,4.253057,0.069275,0.069648,0.069764,0.055304,0.060445,0.051477,0.067573,0.045804,0.088762,0.075506,0.104659,0.050137,0.046960,0.028691,0.074889,0.041107,0.014410,0.026495,0.017206,0.010976,0.019197,0.014546,0.024091,0.011988,0.015363,0.018286,0.018402,0.017703,0.013215,0.005343,0.021327,0.015179,0.013021,0.015215,0.018268,0.014091,0.013562,0.010740,0.013731,0.013568,0.016135,0.015508,0.026591,0.009498,0.010107,0.006470,0.019173,0.009965,0.023997,0.018881,0.022870,0.022380,0.022837,0.017515,0.021418,0.013916,0.026368,0.025725,0.033165,0.019638,0.013471,0.008589,0.020397,0.007797,0.018019,0.009228,0.011595,0.007970,0.004999,0.008805,0.008503,0.006447,0.028653,0.016177,0.026757,0.003423,0.010280,0.008361,0.014178,0.008270,ID_AAYKAN.fastq,Nasal,ID_AAYKAN
3,448014,124.5,0.5,NaN,NaN,0.529188,0.0,0.203929,0.266883,0.255616,0.273572,1.931284,33.422338,5.076070,0.055281,0.047889,0.055491,0.044704,0.054865,0.075711,0.074520,0.070679,0.049232,0.071115,0.067452,0.068245,0.046199,0.077024,0.060221,0.081374,0.018263,0.010775,0.017159,0.009046,0.007113,0.014898,0.012652,0.013616,0.009215,0.014861,0.020643,0.011138,0.007744,0.012551,0.004707,0.020053,0.004272,0.021157,0.016460,0.013410,0.025223,0.016028,0.012923,0.022155,0.004418,0.030490,0.013117,0.027099,0.006994,0.020627,0.018719,0.021358,0.020723,0.003880,0.011733,0.011575,0.008092,0.01165

CPU times: user 137 ms, sys: 14.2 ms, total: 151 ms
Wall time: 151 ms


In [ ]:
train_df['ID']=train_df['filename'].apply(lambda x: x.split('.')[:1][0])
train_df.sort_values(by='ID',inplace=True)
train_df.drop(columns='filename',inplace=True)

df_train=df_train.merge(train_df,on='ID',how='left')
df_train

,num_reads,avg_read_len,read_len_std,read_len_min,read_len_max,gc_content,gc_std,A,T,G,C,avg_entropy,avg_quality,std_quality,AA_freq,AC_freq,AG_freq,AT_freq,CA_freq,CC_freq,CG_freq,CT_freq,GA_freq,GC_freq,GG_freq,GT_freq,TA_freq,TC_freq,TG_freq,TT_freq,AAA_freq,AAC_freq,AAG_freq,AAT_freq,ACA_freq,ACC_freq,ACG_freq,ACT_freq,AGA_freq,AGC_freq,AGG_freq,AGT_freq,ATA_freq,ATC_freq,ATG_freq,ATT_freq,CAA_freq,CAC_freq,CAG_freq,CAT_freq,CCA_freq,CCC_freq,CCG_freq,CCT_freq,CGA_freq,CGC_freq,CGG_freq,CGT_freq,CTA_freq,CTC_freq,CTG_freq,CTT_freq,GAA_freq,GAC_freq,GAG_freq,GAT_freq,GCA_freq,GCC_freq,GCG_freq,GCT_freq,GGA_freq,GGC_freq,GGG_freq,GGT_freq,GTA_freq,GTC_freq,GTG_freq,GTT_freq,TAA_freq,TAC_freq,TAG_freq,TAT_freq,TCA_freq,TCC_freq,TCG_freq,TCT_freq,TGA_freq,TGC_freq,TGG_freq,TGT_freq,TTA_freq,TTC_freq,TTG_freq,TTT_freq,filename,label,ID,SampleType,SubjectID,SampleID
0,42840,124.5,0.5,NaN,NaN,0.552629,0.0,0.182194,0.265177,0.268545,0.284083,1.920269,33.814612,5.260114,0.042977,0.042083,0.053192,0.042267,0.048329,0.086290,0.077074,0.074589,0.053084,0.072132,0.076118,0.068617,0.039265,0.081836,0.064273,0.077874,0.011399,0.009149,0.013774,0.008986,0.003884,0.015486,0.012330,0.010724,0.008209,0.013395,0.024267,0.007750,0.007140,0.013177,0.002331,0.019952,0.002524,0.018687,0.016994,0.010508,0.026537,0.022126,0.012268,0.026061,0.006587,0.031601,0.012660,0.026838,0.002638,0.025675,0.020815,0.022066,0.017881,0.004217,0.014414,0.014143,0.006709,0.012391,0.028534,0.025073,0.020147,0.016368,0.021112,0.019107,0.015467,0.013305,0.020158,0.020177,0.011515,0.010369,0.008445,0.008975,0.011589,0.032926,0.024570,0.013337,0.018514,0.011355,0.018700,0.015479,0.010335,0.030344,0.021493,0.016315,ID_AAFNOT.fastq,Mouth,ID_AAFNOT,Skin,Subject_TQDMSG,Sample_PKCWPK
1,42814,400.0,0.0,NaN,NaN,0.539853,0.0,0.268928,0.191218,0.316762,0.223092,1.970735,36.432308,4.110366,0.068139,0.069039,0.075684,0.055772,0.059831,0.050969,0.066888,0.045795,0.090766,0.072336,0.104082,0.049873,0.050760,0.031305,0.068506,0.040255,0.012121,0.025302,0.019065,0.011652,0.018307,0.014046,0.025362,0.011469,0.018427,0.018038,0.021356,0.017933,0.014461,0.006695,0.020634,0.013861,0.015172,0.014303,0.016626,0.013879,0.013895,0.010529,0.012493,0.014112,0.016406,0.015687,0.025184,0.009728,0.011870,0.006757,0.017628,0.009548,0.023510,0.020072,0.025770,0.021247,0.022142,0.017390,0.019544,0.013428,0.027356,0.025066,0.032572,0.019173,0.013679,0.010128,0.016871,0.008967,0.017470,0.009520,0.014413,0.009081,0.005637,0.009132,0.009655,0.006901,0.026414,0.013726,0.025222,0.003164,0.010878,0.007803,0.013543,0.007980,ID_AAXPTO.fastq,Nasal,ID_AAXPTO,Stool,Subject_UDAXIH,Sample_SCJCMW
2,21794,400.0,0.0,NaN,NaN,0.543314,0.0,0.264808,0.191878,0.318555,0.224759,1.971770,36.337955,4.253057,0.069275,0.069648,0.069764,0.055304,0.060445,0.051477,0.067573,0.045804,0.088762,0.075506,0.104659,0.050137,0.046960,0.028691,0.074889,0.041107,0.014410,0.026495,0.017206,0.010976,0.019197,0.014546,0.024091,0.011988,0.015363,0.018286,0.018402,0.017703,0.013215,0.005343,0.021327,0.015179,0.013021,0.015215,0.018268,0.014091,0.013562,0.010740,0.013731,0.013568,0.016135,0.015508,0.026591,0.009498,0.010107,0.006470,0.019173,0.009965,0.023997,0.018881,0.022870,0.022380,0.022837,0.017515,0.021418,0.013916,0.026368,0.025725,0.033165,0.019638,0.013471,0.008589,0.020397,0.007797,0.018019,0.009228,0.011595,0.007970,0.004999,0.008805,0.008503,0.006447,0.028653,0.016177,0.026757,0.003423,0.010280,0.008361,0.014178,0.008270,ID_AAYKAN.fastq,Nasal,ID_AAYKAN,Stool,Subject_YXWGWJ,Sample_PAHSXW
3,448014,124.5,0.5,NaN,NaN,0.529188,0.0,0.203929,0.266883,0.255616,0.273572,1.931284,33.422338,5.076070,0.055281,0.047889,0.055491,0.044704,0.054865,0.075711,0.074520,0.070679,0.049232,0.071115,0.067452,0.068245,0.046199,0.077024,0.060221,0.081374,0.018263,0.010775,0.017159,0.009046,0.007113,0.014898,0.012652,0.013616,0.009215,0.014861,0.020643,0.011138,0.007744,0.012551,0.004707,0.020053,0.004272,0.021157,0.016460,0.013410,0.025223,0.016028,0.012923

# 📊 Feature Engineering

I hand engineered several biologically meaningful and statistical features to enhance classification. Without them, my CV worsened to 0.04 from 0.03. Also the private leaderboard worsens to 0.021 from 0.020:

---

### 1. Nucleotide Balance and Skew Features
- **AT/GC ratio** – balance of AT vs GC bases  
- **Purine/Pyrimidine ratio** – balance of A+G vs C+T  
- **GC skew** – asymmetry between G and C  
- **AT skew** – asymmetry between A and T  

---

### 2. K-mer Diversity Metrics
- **Shannon entropy** of k-mer distribution (k = 2, 3)  
- **Gini coefficient** to capture inequality in k-mer usage  
- **Common k-mer count** (above frequency threshold)  
- **Maximum k-mer frequency**  
- **Coefficient of variation** of k-mer frequencies  

---

### 3. Stop Codon and Biological Significance Features
- Frequencies of **stop codons** (TAA, TAG, TGA)  
- **Total stop codon frequency**  
- **Start codon ATG frequency**  
- **CpG-related ratio** (CG dinucleotide frequency vs C·G product)  

---

### 4. Interaction and Polynomial Features
- Pairwise **interactions** between key base features  
- **Squared terms** and polynomial combinations (degree 2)  
- Captures nonlinear relationships between GC content, read length, quality, and base counts  

---

### 5. Read Length Distribution Features
- **Coefficient of variation** of read length  
- **Read length to quality ratio**  
- **Total sequenced bases** (num_reads × avg_read_len)  

---

### 6. Quality Distribution Features
- **Quality-to-entropy ratio**  
- **Signal-to-noise ratio** (mean/std of quality scores)  

---

✅ These engineered features capture **sequence composition, diversity, functional motifs, statistical variation, and quality signals** that may be relevant for downstream classification.


In [ ]:
#1. Nucleotide Balance and Skew Features
# These capture imbalances between complementary nucleotides, which can be biologically significant.

# Calculate ratios and skews
def add_balance_features(df):
    df = df.copy()
    # AT/GC Ratio
    df['AT_GC_ratio'] = (df['A'] + df['T']) / (df['G'] + df['C']).replace(0, np.nan)

    # Purine (A,G) vs Pyrimidine (C,T) ratio
    df['purine_pyr_ratio'] = (df['A'] + df['G']) / (df['C'] + df['T']).replace(0, np.nan)

    # GC Skew - measures strand asymmetry
    df['GC_skew'] = (df['G'] - df['C']) / (df['G'] + df['C']).replace(0, np.nan)

    # AT Skew
    df['AT_skew'] = (df['A'] - df['T']) / (df['A'] + df['T']).replace(0, np.nan)

    # Fill any NaN values created by division by zero
    df = df.fillna(0)
    return df

# Apply to both dataframes
#df_train = add_balance_features(df_train)
#df_test = add_balance_features(df_test)

def gini(array):
    """Compute Gini coefficient of a numpy array."""
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0:
        array -= np.amin(array)  # make all values non-negative
    array += 1e-8  # avoid division by zero
    array = np.sort(array)
    n = array.size
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array))


# 2. K-mer Diversity Metrics
# Create features that describe the distribution of k-mers rather than just individual frequencies.

def add_kmer_diversity_features(df, k_values=[2, 3]):
    df = df.copy()

    for k in k_values:
        # Get all k-mer columns for this k value
        kmer_cols = [col for col in df.columns if col.endswith(f'_freq') and len(col.split('_')[0]) == k]

        if not kmer_cols:
            continue

        kmer_matrix = df[kmer_cols].values

        # 1. Shannon Entropy of k-mer distribution
        df[f'{k}mer_entropy'] = [entropy(row) for row in kmer_matrix]

        # 2. Gini Impurity (measure of inequality in distribution)
        df[f'{k}mer_gini'] = [gini(row) for row in kmer_matrix]

        # 3. Number of "common" k-mers (above threshold)
        df[f'{k}mer_common_count'] = (kmer_matrix > 0.01).sum(axis=1)

        # 4. Maximum k-mer frequency
        df[f'{k}mer_max_freq'] = kmer_matrix.max(axis=1)

        # 5. Coefficient of variation of k-mer frequencies
        with np.errstate(divide='ignore', invalid='ignore'):
            cv = np.std(kmer_matrix, axis=1) / np.mean(kmer_matrix, axis=1)
        df[f'{k}mer_cv'] = np.nan_to_num(cv, nan=0.0, posinf=0.0, neginf=0.0)

    return df

#df_train = add_kmer_diversity_features(df_train)
#df_test = add_kmer_diversity_features(df_test)

# 3. Stop Codon and Biological Significance Features
def add_biological_features(df):
    df = df.copy()

    # Stop codon frequencies (TAA, TAG, TGA)
    stop_codons = ['TAA', 'TAG', 'TGA']
    for codon in stop_codons:
        if f'{codon}_freq' in df.columns:
            df[f'stop_{codon}'] = df[f'{codon}_freq']

    # Total stop codon frequency
    stop_cols = [f'stop_{codon}' for codon in stop_codons if f'stop_{codon}' in df.columns]
    if stop_cols:
        df['total_stop_freq'] = df[stop_cols].sum(axis=1)

    # Start codon frequency (ATG)
    if 'ATG_freq' in df.columns:
        df['start_ATG_freq'] = df['ATG_freq']

    # CpG island related (CG frequency)
    if 'CG_freq' in df.columns:
        df['CG_ratio'] = df['CG_freq'] / (df['C'] * df['G']).replace(0, np.nan)
        df['CG_ratio'] = df['CG_ratio'].fillna(0)

    return df

#df_train = add_biological_features(df_train)
#df_test = add_biological_features(df_test)

# 4. Interaction and Polynomial Features
# Create features that capture interactions between your most important base features.

def add_interaction_features(df):
    df = df.copy()

    # Select base features for interactions
    base_features = ['gc_content', 'avg_read_len', 'avg_quality', 'avg_entropy',
                    'num_reads', 'A', 'T', 'G', 'C']

    # Keep only features that exist in dataframe
    base_features = [f for f in base_features if f in df.columns]

    if base_features:
        # Create polynomial features (degree 2)
        poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
        poly_features = poly.fit_transform(df[base_features])

        # Create meaningful names
        feature_names = []
        for i, feat in enumerate(base_features):
            for j, feat2 in enumerate(base_features):
                if i <= j:  # Avoid duplicates
                    if i == j:
                        feature_names.append(f'{feat}^2')
                    else:
                        feature_names.append(f'{feat}*{feat2}')

        # Add to dataframe
        poly_df = pd.DataFrame(poly_features[:, len(base_features):],
                              columns=feature_names,
                              index=df.index)
        df = pd.concat([df, poly_df], axis=1)

    return df

#df_train = add_interaction_features(df_train)
#df_test = add_interaction_features(df_test)

# 5. Read Length Distribution Features
# Enhance your read length statistics.

def add_read_length_features(df):
    df = df.copy()

    # Coefficient of variation of read length
    df['read_len_cv'] = df['read_len_std'] / df['avg_read_len'].replace(0, np.nan)
    df['read_len_cv'] = df['read_len_cv'].fillna(0)

    # Read length to quality ratio
    df['read_len_quality_ratio'] = df['avg_read_len'] / df['avg_quality'].replace(0, np.nan)
    df['read_len_quality_ratio'] = df['read_len_quality_ratio'].fillna(0)

    # Total sequenced bases
    df['total_bases'] = df['num_reads'] * df['avg_read_len']

    return df

#df_train = add_read_length_features(df_train)
#df_test = add_read_length_features(df_test)

# 6. Quality Distribution Features
def add_quality_features(df):
    df = df.copy()

    # Quality to entropy ratio
    df['quality_entropy_ratio'] = df['avg_quality'] / df['avg_entropy'].replace(0, np.nan)
    df['quality_entropy_ratio'] = df['quality_entropy_ratio'].fillna(0)

    # Signal to noise ratio (mean/std of quality)
    df['quality_snr'] = df['avg_quality'] / df['std_quality'].replace(0, np.nan)
    df['quality_snr'] = df['quality_snr'].fillna(0)

    return df

#df_train = add_quality_features(df_train)
#df_test = add_quality_features(df_test)

In [ ]:
#7. Apply All Feature Engineering
#Apply all feature engineering steps in sequence

def engineer_all_features(df):
    df = add_balance_features(df)
    df = add_kmer_diversity_features(df)
    df = add_biological_features(df)
    df = add_read_length_features(df)
    df = add_quality_features(df)
    #df = add_interaction_features(df)  # Do this last as it creates many features
    return df

# Apply to both datasets
df_train = engineer_all_features(df_train)
df_test = engineer_all_features(df_test)
#print(f"Original features: {len(df_train.columns)}")
#print(f"Enhanced features: {len(df_train_enhanced.columns)}")
#print(f"New features added: {len(df_train_enhanced.columns) - len(df_train.columns)}")

# Pre Processing

In [ ]:
# Encode string labels to integers
df_train["label_enc"] = le.fit_transform(df_train["SampleType"])

In [ ]:
exclude_columns=['SampleType', 'SubjectID', 'read_len_min', 'ID', 'filename',
                 'label', 'SampleID', 'read_len_max', 'label_enc']

use_cols=[col for col in df_train.columns if col not in exclude_columns]

# sanity check: what’s left compared to original
diff_cols = set(df_train.columns) - set(use_cols)
print("Excluded columns:", diff_cols,"\nTotal Columns to be used:",len(use_cols))
print('\n',use_cols)

Excluded columns: {'SampleID', 'label', 'filename', 'read_len_max', 'label_enc', 'SubjectID', 'read_len_min', 'ID', 'SampleType'} 
Total Columns to be used: 117

 ['num_reads', 'avg_read_len', 'read_len_std', 'gc_content', 'gc_std', 'A', 'T', 'G', 'C', 'avg_entropy', 'avg_quality', 'std_quality', 'AA_freq', 'AC_freq', 'AG_freq', 'AT_freq', 'CA_freq', 'CC_freq', 'CG_freq', 'CT_freq', 'GA_freq', 'GC_freq', 'GG_freq', 'GT_freq', 'TA_freq', 'TC_freq', 'TG_freq', 'TT_freq', 'AAA_freq', 'AAC_freq', 'AAG_freq', 'AAT_freq', 'ACA_freq', 'ACC_freq', 'ACG_freq', 'ACT_freq', 'AGA_freq', 'AGC_freq', 'AGG_freq', 'AGT_freq', 'ATA_freq', 'ATC_freq', 'ATG_freq', 'ATT_freq', 'CAA_freq', 'CAC_freq', 'CAG_freq', 'CAT_freq', 'CCA_freq', 'CCC_freq', 'CCG_freq', 'CCT_freq', 'CGA_freq', 'CGC_freq', 'CGG_freq', 'CGT_freq', 'CTA_freq', 'CTC_freq', 'CTG_freq', 'CTT_freq', 'GAA_freq', 'GAC_freq', 'GAG_freq', 'GAT_freq', 'GCA_freq', 'GCC_freq', 'GCG_freq', 'GCT_freq', 'GGA_freq', 'GGC_freq', 'GGG_freq', 'GGT_freq'

In [ ]:
# Define features & target
X = df_train[use_cols]
y = df_train["label_enc"]

# Modelling

In [ ]:
%%time
if Params['Tune']:
    n_splits = 10
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    def objective(trial):
        params = {
            "objective": "multiclass" if len(np.unique(y)) > 2 else "binary",
            "num_class": len(np.unique(y)) if len(np.unique(y)) > 2 else 1,
            "metric": "multi_logloss" if len(np.unique(y)) > 2 else "binary_logloss",
            "verbose": -1,
            "random_state": 42,
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                        #early_stopping_rounds=50,
                        #verbose_eval=False

        }

        cv_scores = []

        for train_idx, val_idx in skf.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            lgb_train = lgb.Dataset(X_train, y_train)
            lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

            model = lgb.train(
                params,
                lgb_train,
                valid_sets=[lgb_val],
                num_boost_round=1000,
            )

            preds = model.predict(X_val, num_iteration=model.best_iteration)

            if params["objective"] == "binary":
                # LightGBM outputs probs for class 1 → shape (n_samples,)
                log_losses = log_loss(y_val, np.vstack([1 - preds, preds]).T)
            else:
                # Multiclass already outputs (n_samples, n_classes)
                log_losses = log_loss(y_val, preds)

            cv_scores.append(log_losses)


        return np.mean(cv_scores)

    # Run Optuna
    study = optuna.create_study(direction="minimize")  # "minimize" if using logloss
    study.optimize(objective, n_trials=15)

    print("Best trial:")
    trial = study.best_trial
    print(trial.params)



#Best trial (12):
#{'learning_rate': 0.005651501196508746, 'num_leaves': 197, 'feature_fraction': 0.7609781399502167,
#'bagging_fraction': 0.9988081394675064, 'bagging_freq': 6, 'min_child_samples': 34}

CPU times: user 5 µs, sys: 1e+03 ns, total: 6 µs
Wall time: 10.7 µs


In [ ]:
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))  # Out-of-fold predictions

models = []

#tracker=EmissionsTracker()
#tracker.start()

#stratify_labels = df_train_final['Class']
#for fold, (train_idx, val_idx) in enumerate(skf.split(X, stratify_labels)):
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"===== Fold {fold+1} =====")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM dataset
    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

    # Parameters (tune these for your problem)
    params = {
        "objective": "multiclass" if len(np.unique(y)) > 2 else "binary",
        "num_class": len(np.unique(y)) if len(np.unique(y)) > 2 else 1,
        "metric": "multi_logloss" if len(np.unique(y)) > 2 else "binary_logloss",
        "verbose": -1,
        "random_state": 42,
        'verbose_eval':100,

        #Tuned 15 trials
        'learning_rate': 0.005651501196508746, 'num_leaves': 197, 'feature_fraction': 0.7609781399502167,
        'bagging_fraction': 0.9988081394675064, 'bagging_freq': 6, 'min_child_samples': 34

    }

    # Train
    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_val],
        num_boost_round=1000,

    )

    models.append(model)
    # Predict probabilities for log loss
    preds_proba = model.predict(X_val, num_iteration=model.best_iteration)

    # Predict on validation fold
    if params["objective"] == "binary":
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        preds = (preds > 0.5).astype(int)
    else:  # multiclass
        preds = np.argmax(model.predict(X_val, num_iteration=model.best_iteration), axis=1)

    oof_preds[val_idx] = preds

    # 🔑 log loss instead of accuracy
    fold_logloss = log_loss(y_val, preds_proba)
    print(f"Fold {fold+1} Log Loss: {fold_logloss:.4f}")

    # After predicting
    #print(classification_report(y_val, preds, target_names=le.classes_))

#emissions: float=tracker.stop()

# Overall CV log loss
oof_preds_proba = np.zeros((len(y), preds_proba.shape[1])) if params["objective"] == "multiclass" else np.zeros(len(y))

#for fold, (train_idx, val_idx) in enumerate(skf.split(X, stratify_labels)):
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    # recompute probabilities for OOF log loss aggregation
    model = models[fold]
    preds_proba = model.predict(X.iloc[val_idx], num_iteration=model.best_iteration)
    oof_preds_proba[val_idx] = preds_proba

overall_logloss = log_loss(y, oof_preds_proba)
print(f"\nOverall CV Log Loss: {overall_logloss:.4f}")

===== Fold 1 =====
Fold 1 Log Loss: 0.0923
===== Fold 2 =====
Fold 2 Log Loss: 0.0025
===== Fold 3 =====
Fold 3 Log Loss: 0.0842
===== Fold 4 =====
Fold 4 Log Loss: 0.0033
===== Fold 5 =====
Fold 5 Log Loss: 0.1037
===== Fold 6 =====
Fold 6 Log Loss: 0.0035
===== Fold 7 =====
Fold 7 Log Loss: 0.0091
===== Fold 8 =====
Fold 8 Log Loss: 0.0074
===== Fold 9 =====
Fold 9 Log Loss: 0.0397
===== Fold 10 =====
Fold 10 Log Loss: 0.0290

Overall CV Log Loss: 0.0375


In [ ]:
# # Use the last trained model or an ensemble
# explainer = shap.TreeExplainer(models[-1])   # take the last fold model
# # or you could ensemble shap values from multiple folds if you want

# shap_values = explainer.shap_values(X)

# # SHAP values from explainer
# shap_values = explainer.shap_values(X)  # list of arrays, one per class

# if isinstance(shap_values, list):
#     n_classes = len(shap_values)
#     class_names = le.inverse_transform(np.arange(n_classes))  # use label encoder if available

#     # Per-class mean |SHAP|
#     mean_abs_shap = [np.abs(sv).mean(axis=0) for sv in shap_values]

#     # Rank features by overall importance (average across classes)
#     overall_importance = np.mean(mean_abs_shap, axis=0)
#     top_idx = np.argsort(overall_importance)[-10:]  # top 10
#     top_features = X.columns[top_idx]

#     # Plot
#     fig, ax = plt.subplots(figsize=(10, 6))
#     ind = np.arange(len(top_idx))  # only top features
#     width = 0.8 / n_classes

#     for i, class_name in enumerate(class_names):
#         ax.barh(
#             ind + i * width,
#             mean_abs_shap[i][top_idx],
#             height=width,
#             label=f"Class {class_name}"
#         )

#     ax.set_yticks(ind + width * (n_classes - 1) / 2)
#     ax.set_yticklabels(top_features)
#     ax.invert_yaxis()
#     ax.set_xlabel("Mean |SHAP value|")
#     ax.set_title("Top 10 per-class SHAP feature importance")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()
# else:
#     print("Binary classification detected — use regular shap.summary_plot instead.")

In [ ]:
# if isinstance(shap_values, list):
#     # Take mean absolute SHAP values across classes
#     shap_values_mean = np.mean([np.abs(sv) for sv in shap_values], axis=0)
# else:
#     # Binary classification → shap_values is already an array
#     shap_values_mean = shap_values

In [ ]:
# shap.summary_plot(shap_values_mean, X, max_display=10)

In [ ]:
# # Summary plot (mean |SHAP| values per feature)
# shap.summary_plot(shap_values_mean, X, plot_type="bar", max_display=10)

In [ ]:
# for class_idx, class_name in enumerate(le.classes_):
#     print(f"=== SHAP values for class {class_name} ===")
#     shap.summary_plot(shap_values[class_idx], X, max_display=10, show=False)
#     plt.title(f"Class: {class_name}")
#     plt.show()

# 🔬 Model Insights from SHAP Analysis

After tuning and training the model on extracted FASTQ-derived features and running SHAP interpretability, the following insights emerged:

---

## 1. Feature Importance Across Classes
- The **per-class SHAP bar plot** highlights that different genomic/k-mer features drive predictions for different sample origins (Mouth, Nasal, Skin, Stool).  
- **Examples:**
  - `TCT_freq` and `TTA_freq` are strong predictors for **Mouth** samples.  
  - `ATT_freq` and `CC_freq` dominate in **Nasal** classification.  
  - `TCC_freq` and `CTC_freq` contribute heavily to **Skin**.  
  - `CT_freq` and `AT_skew` are key for **Stool**.  

👉 This suggests that the classifier leverages **distinct sequence motifs** that are biologically tied to different environments.

---

## 2. Overall Feature Stability
- The **SHAP summary plot** (top 10 features overall) confirms that:
  - Features like `AT_skew`, `CT_freq`, and `TCT_freq` consistently impact the model across all samples.  
  - The **direction of SHAP values** shows how high/low feature values push predictions toward different classes.  
  - Example: High `AT_skew` tends to shift predictions strongly toward specific classes, while low values push toward others.

---

## 3. Biological Interpretability
- The dominance of **k-mer frequencies** (di- and tri-nucleotides) aligns with the idea that **genomic composition biases** can serve as biomarkers for microbial community origins.  
- Features like `GC`-related patterns (`CC_freq`, `CT_freq`, `TCC_freq`) point to **genome stability/complexity differences** across environments.  
- `AT_skew` (strand bias) appears as a **global driver**, indicating systematic sequence composition differences.

---

## 4. Practical Implications
- The model is not relying on noise but rather interpretable sequence-based features.  
- **Per-class interpretability** makes the model actionable:
  - For diagnostics, specific k-mer signals can be tied to microbial communities.  
  - For validation, future datasets can be tested for the recurrence of these top motifs.  

---

✅ **Conclusion:**  
The combination of **per-class SHAP bar plots** and the **summary distribution plot** provides both a **global view of model drivers** and a **class-specific signature** of k-mers. This strengthens confidence in the model’s biological relevance and suggests meaningful downstream hypotheses for further study.

# Inference

In [ ]:
# Predict
X_test = df_test[use_cols]

n_classes = len(np.unique(y))
proba_product = np.ones((len(X_test), n_classes))

for model in models:
    preds = model.predict(X_test, num_iteration=model.best_iteration)
    proba_product *= preds

# averaged probabilities from multiple models
proba_geom = proba_product ** (1/len(models))
proba_geom = proba_geom / proba_geom.sum(axis=1, keepdims=True)
y_new_pred = np.argmax(proba_geom, axis=1)

# Create output DataFrame
y_new_pred_labels = le.inverse_transform(y_new_pred)
print(y_new_pred_labels[:10])  # first 10 predicted classes
class_labels = le.inverse_transform(np.arange(n_classes))
probs_df = pd.DataFrame(proba_geom, columns=class_labels)  # class labels as column names
probs_df.insert(0, "filename", df_test["filename"])
probs_df["filename"] = probs_df["filename"].str.replace(".fastq", "", regex=False)

# Save to CSV
probs_df.to_csv("all_feats_tuned.csv", index=False)
probs_df.head()

['Nasal' 'Nasal' 'Mouth' 'Mouth' 'Mouth' 'Skin' 'Stool' 'Stool' 'Stool'
 'Mouth']


,filename,Mouth,Nasal,Skin,Stool
0,ID_ABHFUP,0.000058,0.999740,0.000073,0.000129
1,ID_ADBLNY,0.000061,0.999784,0.000076,0.000079
2,ID_AFAEMB,0.999163,0.000123,0.000577,0.000137
3,ID_AFBBWK,0.999724,0.000082,0.000102,0.000092
4,ID_AGHEZK,0.999186,0.000174,0.000447,0.000193


# End

In [ ]:
emissions_kg = tracker.stop()
print(f"Emissions: {emissions_kg:.6f} kg CO₂e")